# Cross-encoder pair scorer (Colab A100/H100)

Fine-tunes `xlm-roberta-base` (MIT, 278M params) on labelled candidate pairs and scores every kept candidate pair.
Its score becomes the `ce_logit` feature of the LightGBM matcher.

**Upload 1 (before the training cells):** after the laptop runs `python -m src.ce_data train`, copy `work/ce/` to Google
Drive as `MyDrive/ber_ce/` (`ce_pairs.parquet`, `work_pairs.parquet`, `records_train.parquet`, `cross_encoder.py`).

**Upload 2 (before the test cell):** after `python -m src.ce_data test`, add `test_pairs.parquet` and `records_test.parquet`
to the same Drive folder.

**After running:** copy `work_scores.parquet` and `test_scores.parquet` from `MyDrive/ber_ce/` back into `work/ce/`.

Run the cells top to bottom. If `drive.mount` does not work through the VS Code extension, open this notebook in
the browser at colab.research.google.com instead; nothing else changes.

In [1]:
print("Kernel is responding", flush=True)
!timeout 15s nvidia-smi --query-gpu=name,memory.total --format=csv
print("Importing torch...", flush=True)
import torch
print("torch:", torch.__version__, flush=True)

print("Importing transformers...", flush=True)
import transformers
print("transformers:", transformers.__version__, flush=True)

print("Checking CUDA...", flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)

Kernel is responding
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
Importing torch...
torch: 2.11.0+cu128
Importing transformers...
transformers: 5.16.1
Checking CUDA...
CUDA available: True


In [2]:
!nproc; free -g; nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /content | tail -1

48
               total        used        free      shared  buff/cache   available
Mem:             176           2         170           0           4         174
Swap:              0           0           0
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
overlay         113G   48G   66G  43% /


In [4]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ber_ce'
LOCAL = '/content/ce'
!ls -la {DRIVE}

Mounted at /content/drive
total 329349
drwx------ 2 root root      4096 Sep 26 11:42 .
drwx------ 4 root root      4096 Sep 26 12:27 ..
-rw------- 1 root root  16158653 Sep 26 12:00 ce_pairs.parquet
-rw------- 1 root root      7468 Sep 26 11:56 cross_encoder.py
-rw------- 1 root root 293425968 Sep 26 12:01 records_train.parquet
-rw------- 1 root root  27651772 Sep 26 12:00 work_pairs.parquet


In [5]:
# Local SSD is much faster than reading parquet from Drive during training.
!mkdir -p {LOCAL} && cp {DRIVE}/ce_pairs.parquet {DRIVE}/work_pairs.parquet {DRIVE}/records_train.parquet {DRIVE}/cross_encoder.py {LOCAL}/ && ls -la {LOCAL}

total 329352
drwxr-xr-x 2 root root      4096 Sep 26 12:28 .
drwxr-xr-x 1 root root      4096 Sep 26 12:27 ..
-rw------- 1 root root  16158653 Sep 26 12:27 ce_pairs.parquet
-rw------- 1 root root      7468 Sep 26 12:28 cross_encoder.py
-rw------- 1 root root 293425968 Sep 26 12:28 records_train.parquet
-rw------- 1 root root  27651772 Sep 26 12:27 work_pairs.parquet


In [6]:
# Train: 1 epoch over the cross-encoder pairs (entities disjoint from the LightGBM work set).
# Prints throughput every 200 steps and held-out AUC / log-loss at the end.
!python {LOCAL}/cross_encoder.py train --data {LOCAL} --model xlm-roberta-base --out {LOCAL}/ce_model --batch 256 --workers 8
!cp -r {LOCAL}/ce_model {DRIVE}/

train pairs 1,681,144 (positive 41.9%) | held-out 52,156
config.json: 100% 615/615 [00:00<00:00, 6.63MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 337kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 40.8MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 48.5MB/s]

model.safetensors: downloading bytes:  10% 116M/1.12G [00:00<00:03, 265MB/s, 6.61MB/s  ]  
model.safetensors: downloading bytes:  14% 159M/1.12G [00:00<00:03, 313MB/s, 11.4MB/s  ]
model.safetensors: downloading bytes:  28% 312M/1.12G [00:01<00:02, 385MB/s, 25.0MB/s  ] ]
model.safetensors: downloading bytes:  52% 578M/1.12G [00:01<00:01, 420MB/s, 51.3MB/s  ] ]
model.safetensors: downloading bytes:  59% 663M/1.12G [00:01<00:01, 422MB/s, 55.9MB/s  ] ]
model.safetensors: downloading bytes:  65% 728M/1.12G [00:02<00:00, 389MB/s, 63.5MB/s  ] ]
model.safetensors: downloading bytes: 100% 745M/745M [00:02<00:00, 272MB/s, 64.7MB/s  ]   ]
model.safetensors: reconstructing file: 100% 1.12G/1.12G [00:02<00:00, 4

In [7]:
# Score the LightGBM work-set pairs (used to train the matcher with the ce_logit feature).
!python {LOCAL}/cross_encoder.py score --data {LOCAL} --out {LOCAL}/ce_model --pairs work_pairs.parquet --records records_train.parquet --workers 8
!cp {LOCAL}/work_scores.parquet {DRIVE}/

Loading weights: 100% 201/201 [00:00<00:00, 14730.48it/s]
scored 2,966,740 pairs in 4.0 min -> /content/ce/work_scores.parquet


In [8]:
# Score every kept test candidate pair. The test files are exported after the training files, so copy them now.
!cp {DRIVE}/test_pairs.parquet {DRIVE}/records_test.parquet {LOCAL}/
!python {LOCAL}/cross_encoder.py score --data {LOCAL} --out {LOCAL}/ce_model --pairs test_pairs.parquet --records records_test.parquet --workers 8
!cp {LOCAL}/test_scores.parquet {DRIVE}/ && ls -la {DRIVE}

Loading weights: 100% 201/201 [00:00<00:00, 14865.81it/s]
scored 9,853,351 pairs in 13.9 min -> /content/ce/test_scores.parquet
total 1239727
drwx------ 3 root root      4096 Sep 26 13:12 .
drwx------ 5 root root      4096 Sep 26 12:27 ..
drwx------ 2 root root      4096 Sep 26 12:41 ce_model
-rw------- 1 root root  16158653 Sep 26 12:00 ce_pairs.parquet
-rw------- 1 root root      7468 Sep 26 11:56 cross_encoder.py
-rw------- 1 root root 700509113 Sep 26 12:45 records_test.parquet
-rw------- 1 root root 293425968 Sep 26 12:01 records_train.parquet
-rw------- 1 root root  94578883 Sep 26 12:42 test_pairs.parquet
-rw------- 1 root root 106147400 Sep 26 13:12 test_scores.parquet
-rw------- 1 root root  27651772 Sep 26 12:00 work_pairs.parquet
-rw------- 1 root root  30986571 Sep 26 12:46 work_scores.parquet


### Optional: larger model
If time allows after the base model is validated, repeat the three cells above with
`--model BAAI/bge-reranker-v2-m3 --batch 128` (Apache-2.0, ~568M params, about 3x slower) and a different `--out`.